In [1]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    prefix = 'https://codeload.github.com'
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()
        if not (filename_lower.endswith('.md') or filename_lower.endswith('.mdx')):
            continue
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [2]:
my_data = read_repo_data('kubernetes', 'website')
print(f"Documents found: {len(my_data)}")
print(my_data[0])

Error processing website-main/archetypes/blog-post.md: while parsing a block mapping
  in "<unicode string>", line 2, column 1
did not find expected key
  in "<unicode string>", line 3, column 27
Error processing website-main/archetypes/concepts.md: while parsing a block mapping
  in "<unicode string>", line 2, column 1
did not find expected key
  in "<unicode string>", line 2, column 27
Error processing website-main/archetypes/default.md: while parsing a block mapping
  in "<unicode string>", line 2, column 1
did not find expected key
  in "<unicode string>", line 2, column 27
Error processing website-main/archetypes/tasks.md: while parsing a block mapping
  in "<unicode string>", line 2, column 1
did not find expected key
  in "<unicode string>", line 2, column 27
Error processing website-main/archetypes/tutorials.md: while parsing a block mapping
  in "<unicode string>", line 2, column 1
did not find expected key
  in "<unicode string>", line 2, column 27
Documents found: 7501
{'nam

In [3]:
# How many documents?
print(len(my_data))

# Look at the first few
for doc in my_data[:3]:
    print(doc.get('title', 'no title'))
    print(doc['filename'])
    print('---')

7501
no title
website-main/.github/ISSUE_TEMPLATE/bug-report.md
---
no title
website-main/.github/ISSUE_TEMPLATE/feature-request.md
---
no title
website-main/.github/ISSUE_TEMPLATE/support.md
---


In [4]:
# See what metadata fields are available
print(my_data[1].keys())

dict_keys(['name', 'about', 'labels', 'content', 'filename'])


In [5]:
# Find docs about a specific topic
k8s_pods = [doc for doc in my_data if 'pod' in doc.get('title', '').lower()]
print(f"Docs about pods: {len(k8s_pods)}")

Docs about pods: 547


In [6]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [7]:
kuber_chunks = []

for doc in my_data:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    kuber_chunks.extend(chunks)


In [8]:
import re
text = my_data[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())


In [9]:
def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


In [10]:
kuber_chunks = []

for doc in my_data:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        kuber_chunks.append(section_doc)


In [11]:
for doc in my_data[:10]:
    print(doc['filename'], len(doc.get('content', '')))

website-main/.github/ISSUE_TEMPLATE/bug-report.md 510
website-main/.github/ISSUE_TEMPLATE/feature-request.md 560
website-main/.github/ISSUE_TEMPLATE/support.md 457
website-main/.github/PULL_REQUEST_TEMPLATE.md 1292
website-main/CONTRIBUTING.md 2615
website-main/README.md 9284
website-main/SECURITY.md 836
website-main/code-of-conduct.md 147
website-main/content/bn/README.md 10699
website-main/content/bn/_common-resources/index.md 0


In [12]:
my_data[0]

{'name': 'Bug Report',
 'about': 'Report a bug encountered with Kubernetes Website',
 'labels': ['kind/bug'],
 'content': '**This is a Bug Report**\n\n<!-- Thanks for filing an issue! Before submitting, please fill in the following information. -->\n<!-- See https://kubernetes.io/docs/contribute/start/ for guidance on writing an actionable issue description. -->\n\n<!--Required Information-->\n**Problem:**\n\n**Proposed Solution:**\n\n**Page to Update:**\nhttps://kubernetes.io/...\n\n<!--Optional Information (remove the comment tags around information you would like to include)-->\n<!--Kubernetes Version:-->\n\n<!--Additional Information:-->',
 'filename': 'website-main/.github/ISSUE_TEMPLATE/bug-report.md'}

In [13]:
kuber_docs = sorted(my_data, key=lambda doc: len(doc.get('content', '')))

In [14]:
kuber_copy = my_data[5].copy()
copy_content = kuber_copy.pop('content')


In [15]:
sliding_window(copy_content,2000,1000);

In [16]:
def hybrid_chunking(doc, min_size=100, max_size=2000, window_size=2000, step=1000):
    doc_copy = doc.copy()
    content = doc_copy.pop('content')
    
    sections = split_markdown_by_level(content, level=2)
    
    # if no sections were found, fall back to sliding window on the whole doc
    if not sections:
        chunks = sliding_window(content, window_size, step)
        result = []
        for chunk in chunks:
            c = doc_copy.copy()
            c['chunk'] = chunk['chunk']
            c['method'] = 'sliding_window'
            result.append(c)
        return result
    
    result = []
    pending = []  # accumulates sections that are too short

    for section in sections:
        if len(section) < min_size:
            # too short — hold it and merge with the next section
            pending.append(section)
        elif len(section) > max_size:
            # flush any pending short sections first
            if pending:
                merged = '\n\n'.join(pending)
                c = doc_copy.copy()
                c['chunk'] = merged
                c['method'] = 'merged_short_sections'
                result.append(c)
                pending = []
            # too long — apply sliding window
            chunks = sliding_window(section, window_size, step)
            for chunk in chunks:
                c = doc_copy.copy()
                c['chunk'] = chunk['chunk']
                c['method'] = 'sliding_window'
                result.append(c)
        else:
            # just right — merge any pending short sections with this one
            if pending:
                pending.append(section)
                merged = '\n\n'.join(pending)
                c = doc_copy.copy()
                c['chunk'] = merged
                c['method'] = 'section_merged'
                pending = []
            else:
                c = doc_copy.copy()
                c['chunk'] = section
                c['method'] = 'section'
            result.append(c)
    
    # flush any remaining pending sections
    if pending:
        merged = '\n\n'.join(pending)
        c = doc_copy.copy()
        c['chunk'] = merged
        c['method'] = 'merged_short_sections'
        result.append(c)
    
    return result

In [17]:
hybrid_chunking(my_data[5]);

In [19]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()

In [20]:
def intelligent_chonking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [21]:
# from tqdm.auto import tqdm

# smart_chonks = []

# for doc in tqdm(my_data):
#     doc_copy = doc.copy()
#     doc_content = doc_copy.pop('content')

#     sections = intelligent_chonking(doc_content)
#     for section in sections:
#         section_doc = doc_copy.copy()
#         section_doc['section'] = section
#         smart_chonks.append(section_doc)


In [22]:
from tqdm.auto import tqdm

kuber_chonks = []
for doc in tqdm(my_data):
    chonks = hybrid_chunking(doc)
    kuber_chonks.extend(chonks)

print(f"Total chunks: {len(kuber_chonks)}")

  0%|          | 0/7501 [00:00<?, ?it/s]

Total chunks: 51398


In [23]:
from minsearch import Index

index = Index(
    text_fields=["chunk", "title", "filename"],
    keyword_fields=[]
)
index.fit(kuber_chonks)

In [24]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [25]:
from minsearch import VectorSearch

embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

kuber_embeddings = []
for d in tqdm(kuber_chonks):
    v = embedding_model.encode(d['chunk'])
    kuber_embeddings.append(v)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/51398 [00:00<?, ?it/s]

NameError: name 'np' is not defined

In [26]:

kuber_embeddings = np.array(kuber_embeddings)

vindex = VectorSearch()
vindex.fit(kuber_embeddings, kuber_chonks)

In [27]:
def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    q = embedding_model.encode(query)
    return vindex.search(q, num_results=num_results)

def hybrid_search(query, num_results=5):
    text_results = text_search(query, num_results)
    vector_results = vector_search(query, num_results)
    
    seen = set()
    combined = []
    for r in text_results + vector_results:
        key = r.get('filename', '') + str(r.get('start', ''))
        if key not in seen:
            seen.add(key)
            combined.append(r)
    return combined



In [28]:
query = "How do I restart a pod in Kubernetes?"
print("=== Text Search ===")
for r in text_search(query):
    print(r.get('title', 'no title'), '|', r['filename'])

print("\n=== Hybrid Search ===")
for r in hybrid_search(query):
    print(r.get('title', 'no title'), '|', r['filename'])

=== Text Search ===
Kubernetes v1.35: New level of efficiency with in-place Pod restart | website-main/content/en/blog/_posts/2026/restart-all-containers.md
Kubernetes v1.35: New level of efficiency with in-place Pod restart | website-main/content/en/blog/_posts/2026/restart-all-containers.md
Kubernetes v1.35: New level of efficiency with in-place Pod restart | website-main/content/en/blog/_posts/2026/restart-all-containers.md
Kubernetes v1.35: New level of efficiency with in-place Pod restart | website-main/content/en/blog/_posts/2026/restart-all-containers.md
kubectl rollout restart | website-main/content/en/docs/reference/kubectl/generated/kubectl_rollout/kubectl_rollout_restart.md

=== Hybrid Search ===
Kubernetes v1.35: New level of efficiency with in-place Pod restart | website-main/content/en/blog/_posts/2026/restart-all-containers.md
kubectl rollout restart | website-main/content/en/docs/reference/kubectl/generated/kubectl_rollout/kubectl_rollout_restart.md
Kubernetes v1.35: 通过

In [29]:
test_queries = [
    "How do I scale a deployment?",
    "What is a ConfigMap?",
    "How does pod scheduling work?",
]

for query in test_queries:
    print(f"\nQuery: {query}")
    print("TEXT:")
    for r in text_search(query, num_results=3):
        print(" -", r.get('title', 'no title'))
    print("HYBRID:")
    for r in hybrid_search(query, num_results=3):
        print(" -", r.get('title', 'no title'))
    print()


Query: How do I scale a deployment?
TEXT:
 - Deployment
 - Deployment
 - Deployment
HYBRID:
 - Deployment
 - Deployment
 - Deployment
 - Deployments
 - 运行多实例的应用
 - Deployments


Query: What is a ConfigMap?
TEXT:
 - ConfigMap
 - ConfigMap
 - ConfigMap
HYBRID:
 - ConfigMap
 - 配置 Pod 使用 ConfigMap
 - ConfigMap
 - ConfigMap


Query: How does pod scheduling work?
TEXT:
 - Pod Scheduling Readiness
 - Pod Scheduling Readiness
 - Kubernetes 1.26: Pod Scheduling Readiness
HYBRID:
 - Pod Scheduling Readiness
 - Kubernetes 1.26: Pod Scheduling Readiness
 - Kubernetes Scheduler
 - GenericWorkload
 - Kubernetes v1.32: QueueingHint Brings a New Possibility to Optimize Pod Scheduling



In [ ]:
#Hybrid search works best 

# Text search tends to return duplicate chunks from the same document when a keyword is used a lot but misses other relevant documents


In [40]:

user_prompt = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='groq:llama-3.3-70b-versatile',
    input=chat_messages,
)

print(response.output_text)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************X0oA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [41]:
from pydantic_ai import Agent

system_prompt = """
You are a helpful assistant for Kubernetes documentation.

Always search for relevant information before answering.
If the first search doesn't give you enough information, try different search terms.
Make multiple searches if needed to provide comprehensive answers.
If search results are not relevant, say so and provide general guidance.
"""

agent = Agent(
    name="kubernetes_agent",
    instructions=system_prompt,
    tools=[hybrid_search_tool],
    model='groq:llama-3.3-70b-versatile'
)

In [42]:
question = "How do I restart a pod in Kubernetes?"
result = await agent.run(user_prompt=question)
print(result.output)

You can restart a pod in Kubernetes using the `kubectl rollout restart` command. This command will restart all the containers in the pod. You can also use the `--selector` option to specify a label selector to target specific pods. Additionally, Kubernetes v1.35 introduces a new action called `RestartAllContainers` which allows for a fast, in-place restart of the pod when a container exits in a way that matches a rule with this action. 

Here is an example of how to restart a deployment:
```
kubectl rollout restart deployment/nginx
```
And here is an example of how to restart all deployments in a namespace:
```
kubectl rollout restart deployment -n test-namespace
```


In [43]:
for msg in result.new_messages():
    print(msg)
    print("---")

ModelRequest(parts=[UserPromptPart(content='How do I restart a pod in Kubernetes?', timestamp=datetime.datetime(2026, 3, 29, 19, 8, 58, 92226, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 29, 19, 8, 58, 93425, tzinfo=datetime.timezone.utc), instructions="You are a helpful assistant for Kubernetes documentation.\n\nAlways search for relevant information before answering.\nIf the first search doesn't give you enough information, try different search terms.\nMake multiple searches if needed to provide comprehensive answers.\nIf search results are not relevant, say so and provide general guidance.", run_id='10af9fe8-2f88-45e1-a9c4-517dcc02c796')
---
ModelResponse(parts=[ToolCallPart(tool_name='hybrid_search_tool', args='{"query":"restart pod in Kubernetes"}', tool_call_id='j4nnf4v38')], usage=RequestUsage(input_tokens=418, output_tokens=20), model_name='llama-3.3-70b-versatile', timestamp=datetime.datetime(2026, 3, 29, 19, 8, 58, 429955, tzinfo=datetime.timezone.ut

In [44]:
questions = [
    "What is a ConfigMap and how do I use it?",
    "How does pod scheduling work?",
    "What happens when a node runs out of memory?",
]

for q in questions:
    print(f"Q: {q}")
    result = await agent.run(user_prompt=q)
    print(f"A: {result.output}\n")

Q: What is a ConfigMap and how do I use it?
A: A ConfigMap is an API object used to store non-confidential data in key-value pairs. Pods can consume ConfigMaps as environment variables, command-line arguments, or as configuration files in a volume.

Here is an example of a ConfigMap:
```
apiVersion: v1
kind: ConfigMap
metadata:
  name: example-config
data:
  example.property.1: hello
  example.property.2: world
```
You can create a ConfigMap using the `kubectl create configmap` command. For example:
```
kubectl create configmap example-config --from-literal=example.property.1=hello --from-literal=example.property.2=world
```
You can also create a ConfigMap from a file using the `--from-file` option. For example:
```
kubectl create configmap example-config --from-file=example.properties
```
Once you have created a ConfigMap, you can reference it in a Pod definition. For example:
```
apiVersion: v1
kind: Pod
metadata:
  name: example-pod
spec:
  containers:
  - name: example-container
  

In [ ]:
testo